# PINN advantage #1: High-dimensional PDE (the curse of dimensionality)

Solve the Poisson equation on the unit cube in $d$ dimensions:
$$-\Delta u(\mathbf{x}) = f(\mathbf{x}),\qquad \mathbf{x}\in[0,1]^d,\qquad u=g\ \text{on the boundary}.$$

**Why a grid solver fails:** a finite-difference/finite-element grid with $N$ points
per axis needs $N^d$ unknowns. At $d=10$, even a coarse $N=20$ gives $20^{10}\approx
10^{13}$ points — impossible to store or solve. A PINN instead samples random
collocation points, so its cost grows **mildly** with $d$, not exponentially. This is
a problem class where PINNs do something classical CFD fundamentally cannot.

**Manufactured solution (so we can check accuracy):** we choose
$u^\*(\mathbf{x})=\sum_{i=1}^d \sin(\pi x_i)$. Then
$-\Delta u^\* = \pi^2 \sum_i \sin(\pi x_i) = \pi^2 u^\*$, so $f=\pi^2 u^\*$, and $u^\*=0$
on each face of the cube (since $\sin(0)=\sin(\pi)=0$).

In [ ]:
# Cell 1 -- Imports, device, dimension, and the manufactured solution
import time
import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(0); np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

d = 5          # <-- spatial dimension. Try 8, 10, 20 to feel the curse of dimensionality.
PI = np.pi

def u_exact(x):                      # x: (N, d) -> (N, 1)
    return torch.sin(PI * x).sum(dim=1, keepdim=True)

def f_rhs(x):                        # right-hand side  f = -Laplacian(u*) = pi^2 * u*
    return (PI ** 2) * torch.sin(PI * x).sum(dim=1, keepdim=True)

# Dramatize what a grid would cost at this dimension:
for N in (10, 20):
    print(f'A grid with N={N} points/axis in d={d} needs N^d = {N**d:,} unknowns.')

### Key step A — the Laplacian via automatic differentiation
The residual needs $\Delta u = \sum_{i=1}^d \partial^2 u/\partial x_i^2$. We never form a
stencil. Instead we:
1. get the full gradient $\nabla u$ (shape $(N,d)$) in one `autograd.grad` call, then
2. for each dimension $i$, differentiate the $i$-th gradient component again and keep
   its $i$-th entry — that is $\partial^2 u/\partial x_i^2$ — and sum them.

`create_graph=True` keeps the graph alive so these derivatives are themselves
differentiable w.r.t. the network weights (needed for backprop through the loss).

In [ ]:
# Cell 2 -- Network and a general Laplacian operator
class PINN(nn.Module):
    def __init__(self, d, h=64, n_layers=4):
        super().__init__()
        layers = [nn.Linear(d, h), nn.Tanh()]
        for _ in range(n_layers - 1):
            layers += [nn.Linear(h, h), nn.Tanh()]
        layers += [nn.Linear(h, 1)]
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

def laplacian(u, x):
    """sum_i d^2 u / d x_i^2  for u:(N,1), x:(N,d)."""
    grad_u = torch.autograd.grad(u, x, torch.ones_like(u), create_graph=True)[0]  # (N,d)
    lap = torch.zeros_like(u)
    for i in range(x.shape[1]):
        gi = grad_u[:, i:i+1]                                   # d u / d x_i
        d2 = torch.autograd.grad(gi, x, torch.ones_like(gi), create_graph=True)[0][:, i:i+1]
        lap = lap + d2                                          # add d^2 u / d x_i^2
    return lap

model = PINN(d).to(device)
opt = torch.optim.Adam(model.parameters(), lr=2e-3)
mse = nn.MSELoss()

### Key step B — sampling the interior and the (high-dimensional) boundary
- **Interior:** just draw uniform points in $[0,1]^d$. No mesh.
- **Boundary:** the boundary of the cube is the union of $2d$ faces. To land on it, we
  draw an interior point, pick one random coordinate, and pin it to $0$ or $1$. That
  gives a uniform sample over the faces — trivial in any dimension, whereas *meshing*
  a boundary is the expensive part of classical methods.

In [ ]:
# Cell 3 -- Samplers
def sample_interior(n):
    return (torch.rand(n, d, device=device)).requires_grad_(True)

def sample_boundary(n):
    x = torch.rand(n, d, device=device)
    face = torch.randint(0, d, (n,), device=device)         # which coordinate is pinned
    side = torch.randint(0, 2, (n,), device=device).float() # to 0 or 1
    x[torch.arange(n, device=device), face] = side
    return x

In [ ]:
# Cell 4 -- Train: PDE residual on interior + Dirichlet BC on boundary
N_INT, N_BC, EPOCHS = 4000, 1000, 4000

t0 = time.perf_counter()
for e in range(EPOCHS):
    opt.zero_grad()

    xi = sample_interior(N_INT)
    u  = model(xi)
    res = -laplacian(u, xi) - f_rhs(xi)          # PDE: -Lap(u) - f = 0
    loss_pde = mse(res, torch.zeros_like(res))

    xb = sample_boundary(N_BC)
    loss_bc = mse(model(xb), u_exact(xb))        # u = g on boundary

    loss = loss_pde + 20.0 * loss_bc             # weight BC so it is respected
    loss.backward(); opt.step()

    if e % 500 == 0:
        print(f'epoch {e:4d}  pde {loss_pde.item():.2e}  bc {loss_bc.item():.2e}')
if device.type == 'cuda':
    torch.cuda.synchronize()
train_time = time.perf_counter() - t0
print(f'\nTraining time: {train_time:.2f} s  (d={d})')

In [ ]:
# Cell 5 -- Accuracy check on fresh random test points (no grid needed)
with torch.no_grad():
    xt = torch.rand(20000, d, device=device)
    up = model(xt); ue = u_exact(xt)
    rel_l2 = (torch.norm(up - ue) / torch.norm(ue)).item()
    max_err = (up - ue).abs().max().item()
print(f'Relative L2 error : {rel_l2:.3e}')
print(f'Max abs error     : {max_err:.3e}')
print(f'Solution magnitude: u* ranges roughly in [0, {d}] here.')

## Takeaways
- The PINN solved a $d$-dimensional PDE by sampling points, never building an
  $N^d$ grid. Bump `d` to 8, 10, 20 in Cell 1: training cost scales gently, while the
  printed $N^d$ grid size explodes past anything storable.
- The Laplacian came straight from autograd — no stencils, no assembly of a sparse
  matrix.
- This is the headline advantage: for high-dimensional PDEs (finance/Black-Scholes,
  control/Hamilton-Jacobi-Bellman, many-body problems) mesh methods are simply not an
  option, and mesh-free physics-driven learning is.

*Note:* accuracy per dimension is modest compared to a fine 1-D grid solve — the point
here is **feasibility in high $d$**, not beating a grid in low $d$.